# NetPredict-Ngaoundéré
## Internet Connection Quality Prediction and Network Quality Analytics

**Project type:** End-to-end supervised Machine Learning project  
**Location:** Ngaoundéré, Cameroon  
**Language:** Python  
**Primary tools:** Jupyter Notebook, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, Joblib

---

### Project objective

This notebook develops a complete machine learning workflow for analyzing and predicting Internet connection quality from a locally collected survey dataset in Ngaoundéré.

The original field problem was simple but practical: **Internet connection quality changes according to location, time, usage patterns and other network-related factors.** The objective is therefore to transform collected observations into a reproducible analytical and predictive system.

The project follows a professional data science workflow:

1. Load and inspect the raw dataset.
2. Document the dataset and its limitations.
3. Clean and standardize the data.
4. Explore the structure and distributions.
5. Engineer useful predictive features.
6. Define a supervised classification target.
7. Split data into training and test sets.
8. Build preprocessing pipelines.
9. Train multiple baseline and ensemble models.
10. Evaluate them with appropriate metrics.
11. Inspect errors and feature importance.
12. Save the final model and preprocessing pipeline.
13. Build a reusable prediction function.
14. Generate analytical outputs that can later feed a Streamlit dashboard.

> **Important methodological note:** the provided survey is primarily a *self-reported Internet quality survey*. Unless a numerical Mbps measurement column exists, this project must not claim to predict an actual measured bandwidth value. The target is Internet connection **quality**, as reported in the survey.


## 1. Research question

### Main question

> **Can machine learning predict reported Internet connection quality from contextual information collected from Internet users in Ngaoundéré?**

### Secondary questions

- Which variables are most strongly associated with reported connection quality?
- Does connection quality vary across locations?
- Are there temporal patterns in reported network problems?
- Which machine learning models perform best under the same validation procedure?
- Which observations are most difficult for the model to classify?
- Can the trained model be packaged for later use in an interactive application?

### Scope

This is a **prototype research/engineering project**, not a telecom network measurement system. Survey responses describe users' experiences and perceptions; they do not replace objective network measurements such as latency, packet loss, download speed, upload speed, jitter, RSSI or signal-to-noise ratio.


In [ ]:
# ============================================================
# 2. ENVIRONMENT SETUP
# ============================================================

# If a library is missing in your environment, uncomment:
# !pip install pandas numpy matplotlib seaborn scikit-learn joblib openpyxl

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Environment ready.")


In [ ]:
# ============================================================
# 3. PROJECT PATHS
# ============================================================

# The uploaded CSV is used directly here.
DATA_PATH = "/mnt/data/enquete_qualite_internet_ngaoundere_v3-5.csv"

# Local project structure for a future GitHub repository.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

for folder in [RAW_DIR, PROCESSED_DIR, MODELS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)


## 4. Load the raw dataset

The dataset was collected through a survey about Internet connection quality in Ngaoundéré.

We intentionally keep the first inspection separate from preprocessing. This avoids silently changing the source data before understanding what it contains.


In [ ]:
# ============================================================
# 4. LOAD DATA
# ============================================================

df_raw = pd.read_csv(DATA_PATH)

print(f"Rows: {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")
display(df_raw.head())


In [ ]:
# ============================================================
# 5. BASIC DATA AUDIT
# ============================================================

print("Dataset dimensions:", df_raw.shape)
print("\nData types:")
display(df_raw.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = (
    df_raw.isna()
    .sum()
    .to_frame("missing_count")
)
missing["missing_pct"] = (missing["missing_count"] / len(df_raw) * 100).round(2)
display(missing.sort_values("missing_pct", ascending=False))

print("\nDuplicated rows:", df_raw.duplicated().sum())


In [ ]:
# ============================================================
# 6. UNIQUE VALUES AND CARDINALITY
# ============================================================

cardinality = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [df_raw[c].dtype for c in df_raw.columns],
    "n_unique": [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
    "missing": [df_raw[c].isna().sum() for c in df_raw.columns]
}).sort_values("n_unique")

display(cardinality)


In [ ]:
# ============================================================
# 7. STANDARDIZE COLUMN NAMES
# ============================================================

def clean_column_name(name):
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

df = df_raw.copy()
df.columns = [clean_column_name(c) for c in df.columns]

print("Cleaned column names:")
for c in df.columns:
    print(" -", c)


## 8. Identify the prediction target

The survey contains a variable corresponding to **Q7 – Qualité du débit**. We use this reported quality as the classification target.

The target must be known for the training examples but must not be used as an input feature. This is important to prevent **target leakage**.


In [ ]:
# ============================================================
# 8. FIND TARGET COLUMN
# ============================================================

target_candidates = [
    c for c in df.columns
    if "Q7" in c.upper() and "QUALITE" in c.upper()
]

print("Possible target columns:", target_candidates)

if not target_candidates:
    # Fallback search for a quality-related column.
    target_candidates = [
        c for c in df.columns
        if "qualite" in c.lower() and "debit" in c.lower()
    ]

if not target_candidates:
    raise ValueError(
        "Target column could not be detected automatically. "
        "Inspect df.columns and set TARGET manually."
    )

TARGET = target_candidates[0]
print("Selected target:", TARGET)
print("\nTarget distribution:")
display(df[TARGET].value_counts(dropna=False).to_frame("count"))


In [ ]:
# ============================================================
# 9. TARGET QUALITY CHECK
# ============================================================

print("Target data type:", df[TARGET].dtype)
print("Number of target classes:", df[TARGET].nunique(dropna=True))

target_distribution = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis("class")
    .reset_index(name="count")
)

target_distribution["percentage"] = (
    target_distribution["count"] / len(df) * 100
).round(2)

display(target_distribution)

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x=TARGET, order=df[TARGET].value_counts().index)
plt.title("Distribution of Reported Internet Quality")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 10. Data cleaning

Survey datasets often contain inconsistent representations such as:

- leading/trailing spaces;
- different capitalization;
- empty strings;
- explicit values such as `N/A`, `None`, `Unknown`;
- duplicated records.

The cleaning strategy below is conservative: it standardizes text without inventing information.


In [ ]:
# ============================================================
# 10. CONSERVATIVE DATA CLEANING
# ============================================================

df_clean = df.copy()

# Remove exact duplicate records.
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
after = len(df_clean)

print(f"Removed {before - after} exact duplicate rows.")

# Normalize textual missing-value markers.
MISSING_MARKERS = {
    "", " ", "na", "n/a", "nan", "none", "null",
    "non disponible", "indisponible", "unknown"
}

for col in df_clean.select_dtypes(include="object").columns:
    s = df_clean[col].astype(str).str.strip()
    df_clean[col] = s.where(
        ~s.str.lower().isin(MISSING_MARKERS),
        np.nan
    )

print("Cleaning completed.")


In [ ]:
# ============================================================
# 11. STANDARDIZE TEXT CATEGORIES
# ============================================================

for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .replace({"nan": pd.NA})
    )

# Show categorical distributions.
categorical_cols = df_clean.select_dtypes(include=["object", "string", "category"]).columns

for col in categorical_cols:
    print(f"\n### {col}")
    display(df_clean[col].value_counts(dropna=False).head(20).to_frame("count"))


## 12. Date and time feature engineering

If date/time fields are present, we extract:

- year;
- month;
- day of month;
- day of week;
- hour;
- weekend indicator.

These variables can capture temporal patterns without requiring the model to understand raw date strings.


In [ ]:
# ============================================================
# 12. AUTOMATIC DATE/TIME FEATURE ENGINEERING
# ============================================================

def find_columns_by_keywords(columns, keywords):
    return [
        c for c in columns
        if any(k in c.lower() for k in keywords)
    ]

date_candidates = find_columns_by_keywords(
    df_clean.columns,
    ["date", "jour"]
)

time_candidates = find_columns_by_keywords(
    df_clean.columns,
    ["heure", "time"]
)

print("Date candidates:", date_candidates)
print("Time candidates:", time_candidates)

# Parse candidate date columns where useful.
for col in date_candidates:
    parsed = pd.to_datetime(df_clean[col], errors="coerce", dayfirst=True)
    if parsed.notna().sum() >= max(3, int(0.5 * len(df_clean))):
        df_clean[f"{col}_year"] = parsed.dt.year
        df_clean[f"{col}_month"] = parsed.dt.month
        df_clean[f"{col}_dayofweek"] = parsed.dt.dayofweek
        df_clean[f"{col}_is_weekend"] = parsed.dt.dayofweek.isin([5, 6]).astype(int)
        print(f"Extracted date features from: {col}")

# Parse time columns where useful.
for col in time_candidates:
    parsed_time = pd.to_datetime(
        df_clean[col].astype(str),
        errors="coerce"
    )
    if parsed_time.notna().sum() >= max(3, int(0.3 * len(df_clean))):
        df_clean[f"{col}_hour"] = parsed_time.dt.hour
        print(f"Extracted hour feature from: {col}")


## 13. Automatically detect numeric information

A robust pipeline should not assume every numeric-looking column was stored as a numeric dtype. We attempt safe numeric conversion only when a sufficiently large proportion of values can be converted.

This is useful for future versions of the survey if actual measured values such as speed, latency or signal strength are added.


In [ ]:
# ============================================================
# 13. SAFE NUMERIC CONVERSION
# ============================================================

for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        converted = pd.to_numeric(
            df_clean[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )
        valid_ratio = converted.notna().mean()

        if valid_ratio >= 0.80:
            df_clean[col] = converted
            print(f"Converted to numeric: {col}")


## 14. Exploratory Data Analysis (EDA)

EDA is not just for making graphs. It helps us answer practical questions before modeling:

- What does the sample look like?
- Which locations are represented?
- Which access technologies are represented?
- What kinds of network problems are reported?
- Are some classes extremely rare?
- Which variables may contain leakage or identifiers?

Because this is a small survey dataset, conclusions must be treated as exploratory rather than population-level estimates for all Internet users in Ngaoundéré.


In [ ]:
# ============================================================
# 14. NUMERIC SUMMARY
# ============================================================

numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

if numeric_cols:
    display(df_clean[numeric_cols].describe().T)
else:
    print("No numeric columns detected.")


In [ ]:
# ============================================================
# 15. CATEGORICAL SUMMARY
# ============================================================

cat_summary = []

for col in df_clean.select_dtypes(include=["object", "string", "category"]).columns:
    cat_summary.append({
        "column": col,
        "unique_values": df_clean[col].nunique(dropna=True),
        "missing": df_clean[col].isna().sum(),
        "top_value": df_clean[col].mode(dropna=True).iloc[0] if not df_clean[col].mode(dropna=True).empty else None
    })

display(pd.DataFrame(cat_summary).sort_values("unique_values"))


In [ ]:
# ============================================================
# 16. TARGET VS CATEGORICAL FEATURES
# ============================================================

# Automatically select low/moderate-cardinality categorical variables.
candidate_cats = [
    c for c in df_clean.select_dtypes(include=["object", "string", "category"]).columns
    if c != TARGET and 2 <= df_clean[c].nunique(dropna=True) <= 12
]

print("Candidate categorical variables:", candidate_cats)

for col in candidate_cats[:10]:
    plt.figure(figsize=(10, 5))
    order = df_clean[col].value_counts().index
    sns.countplot(data=df_clean, x=col, hue=TARGET, order=order)
    plt.title(f"{TARGET} by {col}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 17. Detect high-cardinality identifiers

Identifiers such as names, telephone numbers, email addresses or exact response IDs should generally not be used as predictive features. They can cause overfitting and may create privacy risks.

The next cell identifies suspicious columns for manual review. **It does not automatically delete them**, because the final decision should be based on the actual meaning of each column.


In [ ]:
# ============================================================
# 17. IDENTIFIER / HIGH-CARDINALITY AUDIT
# ============================================================

high_cardinality_report = []

for col in df_clean.columns:
    nunique = df_clean[col].nunique(dropna=True)
    ratio = nunique / max(len(df_clean), 1)

    suspicious = any(
        token in col.lower()
        for token in [
            "nom", "name", "telephone", "phone", "email",
            "adresse", "address", "id", "identifiant"
        ]
    )

    high_cardinality_report.append({
        "column": col,
        "unique": nunique,
        "unique_ratio": round(ratio, 3),
        "suspicious_identifier_name": suspicious
    })

id_report = pd.DataFrame(high_cardinality_report)
display(
    id_report.sort_values(
        ["suspicious_identifier_name", "unique_ratio"],
        ascending=[False, False]
    )
)


## 18. Leakage audit

A particularly important question is whether some columns contain information that directly reveals the target.

For example, if the target is reported Internet quality and a feature literally contains a later evaluation of the same quality, including it would make the model appear strong without learning a useful relationship.

We therefore maintain an explicit exclusion list for obvious target-like fields and identifiers.


In [ ]:
# ============================================================
# 18. FEATURE EXCLUSION RULES
# ============================================================

EXCLUDE_KEYWORDS = [
    "q7",
    "qualite_debit",
    "target",
    "label",
    "nom",
    "name",
    "telephone",
    "phone",
    "email",
    "adresse",
    "address"
]

excluded_columns = []

for col in df_clean.columns:
    if col == TARGET:
        excluded_columns.append(col)
        continue

    if any(k in col.lower() for k in EXCLUDE_KEYWORDS):
        excluded_columns.append(col)

print("Columns excluded from modeling:")
for col in sorted(set(excluded_columns)):
    print(" -", col)


## 19. Build the modeling dataset

At this point we define:

- `X`: input features;
- `y`: prediction target.

Rows with a missing target cannot be used for supervised training and are removed.

We keep the feature-selection logic explicit so that it is easy to review and reproduce.


In [ ]:
# ============================================================
# 19. X / y CONSTRUCTION
# ============================================================

model_df = df_clean.dropna(subset=[TARGET]).copy()

feature_cols = [
    c for c in model_df.columns
    if c not in set(excluded_columns)
]

X = model_df[feature_cols].copy()
y = model_df[TARGET].astype(str).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_cols)
print("\nClasses:")
display(y.value_counts().to_frame("count"))


In [ ]:
# ============================================================
# 20. REMOVE VERY HIGH-CARDINALITY TEXT FEATURES
# ============================================================

# High-cardinality free-text fields are risky on a small dataset.
# We keep only categorical columns with manageable cardinality.
MAX_CATEGORIES = 20

safe_feature_cols = []

for col in X.columns:
    if X[col].dtype == "object" or str(X[col].dtype).startswith("string"):
        if X[col].nunique(dropna=True) > MAX_CATEGORIES:
            print(f"Excluded high-cardinality feature: {col}")
            continue
    safe_feature_cols.append(col)

X = X[safe_feature_cols].copy()

print("\nFinal feature count:", len(safe_feature_cols))
print(safe_feature_cols)


## 21. Train/test split

We reserve a test set that is not used during model selection.

Because this is a classification task, we attempt a **stratified split** so that the class proportions are preserved where possible.

With a very small dataset, test metrics can vary substantially from split to split. The reported test score should therefore be interpreted together with cross-validation results.


In [ ]:
# ============================================================
# 21. TRAIN / TEST SPLIT
# ============================================================

class_counts = y.value_counts()
can_stratify = class_counts.min() >= 2 and int(len(y) * TEST_SIZE) >= y.nunique()

if can_stratify:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )
else:
    print("Warning: stratification is not reliable with the current class distribution.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

print("Training:", X_train.shape)
print("Testing :", X_test.shape)


In [ ]:
# ============================================================
# 22. DEFINE PREPROCESSING PIPELINE
# ============================================================

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [
    c for c in X_train.columns
    if c not in numeric_features
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessor created.")


# 23. Model training

We compare several models under the same preprocessing strategy.

### Models

**Dummy Classifier**  
A baseline that predicts using a simple strategy. Any useful model should outperform this baseline.

**Logistic Regression**  
A strong and interpretable baseline for multiclass classification.

**Decision Tree**  
A nonlinear model that can learn simple decision rules.

**Random Forest**  
An ensemble of decision trees that can model nonlinear relationships and interactions.

**Extra Trees**  
Another randomized tree ensemble, useful as a comparison.

**Gradient Boosting**  
A sequential ensemble that can capture nonlinear patterns.

The goal is not to declare a universal "best" algorithm. The goal is to measure how different algorithms behave on this specific dataset.


In [ ]:
# ============================================================
# 23. MODEL DEFINITIONS
# ============================================================

models = {
    "Dummy": DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )
}

pipelines = {
    name: Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}

print("Models ready:", list(pipelines))


In [ ]:
# ============================================================
# 24. CROSS-VALIDATION
# ============================================================

# Choose the largest reasonable number of folds based on
# the smallest class count in the training set.
min_class_train = y_train.value_counts().min()

if min_class_train >= 5:
    n_splits = 5
elif min_class_train >= 3:
    n_splits = 3
elif min_class_train >= 2:
    n_splits = 2
else:
    n_splits = None

if n_splits is None:
    print("Not enough samples per class for reliable stratified CV.")
else:
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "macro_precision": "precision_macro",
        "macro_recall": "recall_macro",
        "macro_f1": "f1_macro"
    }

    cv_results = []

    for name, pipeline in pipelines.items():
        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            error_score="raise"
        )

        cv_results.append({
            "model": name,
            "accuracy_mean": scores["test_accuracy"].mean(),
            "accuracy_std": scores["test_accuracy"].std(),
            "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
            "macro_f1_mean": scores["test_macro_f1"].mean(),
            "macro_f1_std": scores["test_macro_f1"].std()
        })

    cv_results_df = pd.DataFrame(cv_results).sort_values(
        "macro_f1_mean",
        ascending=False
    )

    display(cv_results_df.style.format({
        "accuracy_mean": "{:.3f}",
        "accuracy_std": "{:.3f}",
        "balanced_accuracy_mean": "{:.3f}",
        "macro_f1_mean": "{:.3f}",
        "macro_f1_std": "{:.3f}"
    }))


In [ ]:
# ============================================================
# 25. VISUALIZE CROSS-VALIDATION RESULTS
# ============================================================

if "cv_results_df" in globals():
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=cv_results_df,
        x="macro_f1_mean",
        y="model"
    )
    plt.title("Cross-Validation Macro F1 by Model")
    plt.xlabel("Mean Macro F1")
    plt.ylabel("Model")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.show()


## 26. Select a model without hiding uncertainty

For a multiclass problem with potentially imbalanced classes, **macro F1** is useful because it gives each class equal weight.

The notebook uses cross-validation to identify a candidate model, then evaluates that candidate once on the untouched test set.

This is not a claim that the candidate will generalize perfectly. The sample is small, and external validation on new observations would be valuable.


In [ ]:
# ============================================================
# 26. SELECT CANDIDATE MODEL
# ============================================================

if "cv_results_df" in globals():
    candidate_name = cv_results_df.iloc[0]["model"]
else:
    # Safe fallback if CV could not be performed.
    candidate_name = "Logistic Regression"

candidate_pipeline = pipelines[candidate_name]

print("Candidate model selected from validation procedure:", candidate_name)


In [ ]:
# ============================================================
# 27. FIT CANDIDATE MODEL ON TRAINING DATA
# ============================================================

candidate_pipeline.fit(X_train, y_train)

y_pred = candidate_pipeline.predict(X_test)

test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "macro_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0)
}

print("Test set metrics:")
for metric, value in test_metrics.items():
    print(f"{metric:20s}: {value:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# ============================================================
# 28. CONFUSION MATRIX
# ============================================================

labels = sorted(y.unique())

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels
)
plt.title(f"Confusion Matrix — {candidate_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 29. Error analysis

Accuracy alone does not tell us where the model fails.

We therefore create a table containing:

- the original features;
- the true class;
- the predicted class;
- whether the prediction was correct.

This helps us identify patterns that the model struggles to distinguish.


In [ ]:
# ============================================================
# 29. ERROR ANALYSIS TABLE
# ============================================================

error_analysis = X_test.copy()
error_analysis["true_quality"] = y_test.values
error_analysis["predicted_quality"] = y_pred
error_analysis["correct"] = (
    error_analysis["true_quality"] == error_analysis["predicted_quality"]
)

display(
    error_analysis
    .sort_values("correct")
    .head(30)
)

print("Test errors:", (~error_analysis["correct"]).sum())
print("Test observations:", len(error_analysis))


## 30. Feature importance

Tree-based models can expose feature importance after preprocessing.

Because categorical variables are one-hot encoded, one original variable can become many transformed columns.

We aggregate one-hot encoded importance back to the original feature where possible. This gives a more interpretable view of which survey variables contribute most to the model.


In [ ]:
# ============================================================
# 30. FEATURE IMPORTANCE FOR TREE MODELS
# ============================================================

def get_feature_importance(pipeline):
    model = pipeline.named_steps["model"]
    prep = pipeline.named_steps["preprocessor"]

    if not hasattr(model, "feature_importances_"):
        return None

    feature_names = prep.get_feature_names_out()
    importances = model.feature_importances_

    result = pd.DataFrame({
        "transformed_feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    return result

importance_df = get_feature_importance(candidate_pipeline)

if importance_df is not None:
    display(importance_df.head(30))

    plt.figure(figsize=(10, 8))
    top = importance_df.head(20).sort_values("importance")
    sns.barplot(data=top, x="importance", y="transformed_feature")
    plt.title(f"Top Feature Importances — {candidate_name}")
    plt.tight_layout()
    plt.show()
else:
    print(
        f"{candidate_name} does not expose tree feature_importances_. "
        "Permutation importance can be used instead."
    )


## 31. Permutation importance

Permutation importance is model-agnostic. It estimates how much performance decreases when a feature is randomly shuffled.

For small datasets, the result should be interpreted cautiously. It is an explanatory tool, not proof of causation.


In [ ]:
# ============================================================
# 31. PERMUTATION IMPORTANCE
# ============================================================

from sklearn.inspection import permutation_importance

try:
    perm = permutation_importance(
        candidate_pipeline,
        X_test,
        y_test,
        scoring="f1_macro",
        n_repeats=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    perm_df = pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std
    }).sort_values("importance_mean", ascending=False)

    display(perm_df)

    plt.figure(figsize=(10, 8))
    top_perm = perm_df.head(15).sort_values("importance_mean")
    plt.barh(
        top_perm["feature"],
        top_perm["importance_mean"],
        xerr=top_perm["importance_std"]
    )
    plt.title("Permutation Importance on Test Set")
    plt.xlabel("Mean decrease in Macro F1")
    plt.tight_layout()
    plt.show()

except Exception as e:
    print("Permutation importance could not be computed:", e)


## 32. Geographic analysis

If the survey contains a zone/location field, we can inspect reported Internet quality by location.

This is descriptive analysis only. A survey with 100 responses does not automatically establish that one area represents every user or every network condition in that area.


In [ ]:
# ============================================================
# 32. LOCATION COLUMN DETECTION
# ============================================================

location_candidates = [
    c for c in df_clean.columns
    if any(k in c.lower() for k in ["zone", "quartier", "localite", "lieu", "arrondissement", "location"])
]

print("Possible location columns:", location_candidates)


In [ ]:
# ============================================================
# 33. QUALITY BY LOCATION
# ============================================================

if location_candidates:
    LOCATION_COL = location_candidates[0]

    location_table = pd.crosstab(
        df_clean[LOCATION_COL],
        df_clean[TARGET],
        normalize="index"
    ) * 100

    display(location_table.round(1))

    location_table.plot(
        kind="bar",
        stacked=True,
        figsize=(12, 6)
    )
    plt.title("Reported Internet Quality Distribution by Location")
    plt.ylabel("Percentage of responses")
    plt.xlabel(LOCATION_COL)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    LOCATION_COL = None
    print("No location column detected automatically.")


## 34. Temporal analysis

If a time or date variable exists, we can examine whether reported network quality differs across time.

For future fieldwork, this is particularly useful because repeated measurements at fixed locations and fixed time intervals can produce a much stronger temporal dataset.


In [ ]:
# ============================================================
# 34. TEMPORAL ANALYSIS
# ============================================================

hour_cols = [
    c for c in df_clean.columns
    if "hour" in c.lower() or "heure" in c.lower()
]

print("Possible hour columns:", hour_cols)


In [ ]:
# ============================================================
# 35. QUALITY BY HOUR WHEN AVAILABLE
# ============================================================

if hour_cols:
    HOUR_COL = hour_cols[0]

    temp = df_clean[[HOUR_COL, TARGET]].dropna().copy()
    temp[HOUR_COL] = pd.to_numeric(temp[HOUR_COL], errors="coerce")
    temp = temp.dropna()

    if not temp.empty:
        display(
            temp.groupby(HOUR_COL)[TARGET]
            .value_counts(normalize=True)
            .mul(100)
            .rename("percentage")
            .reset_index()
            .head(50)
        )
else:
    HOUR_COL = None
    print("No usable hour feature detected.")


# 36. Hyperparameter tuning

Once the baseline models have been compared, we can tune the main ensemble model.

The grid is deliberately moderate because this is a small dataset. Large hyperparameter searches on a tiny dataset can create a false sense of precision and consume unnecessary compute.


In [ ]:
# ============================================================
# 36. RANDOM FOREST HYPERPARAMETER TUNING
# ============================================================

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 4, 8],
    "model__min_samples_leaf": [1, 2, 4]
}

if n_splits is not None:
    grid_search = GridSearchCV(
        rf_pipeline,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=0
    )

    grid_search.fit(X_train, y_train)

    print("Best parameters:")
    print(grid_search.best_params_)
    print("\nBest CV Macro F1:", round(grid_search.best_score_, 4))

    tuned_rf = grid_search.best_estimator_
else:
    tuned_rf = rf_pipeline.fit(X_train, y_train)
    print("Tuning skipped because the dataset is too small for reliable CV.")


In [ ]:
# ============================================================
# 37. EVALUATE TUNED RANDOM FOREST
# ============================================================

tuned_pred = tuned_rf.predict(X_test)

tuned_metrics = {
    "accuracy": accuracy_score(y_test, tuned_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, tuned_pred),
    "macro_precision": precision_score(y_test, tuned_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_test, tuned_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(y_test, tuned_pred, average="macro", zero_division=0)
}

print("Tuned Random Forest — test metrics")
for metric, value in tuned_metrics.items():
    print(f"{metric:20s}: {value:.4f}")

print("\nClassification report:")
print(classification_report(y_test, tuned_pred, zero_division=0))


In [ ]:
# ============================================================
# 38. FINAL MODEL DECISION
# ============================================================

# We compare the candidate and tuned RF on the same held-out test set.
comparison = pd.DataFrame([
    {"model": candidate_name, **test_metrics},
    {"model": "Tuned Random Forest", **tuned_metrics}
])

display(comparison.sort_values("macro_f1", ascending=False))

# Select using the documented macro-F1 criterion on the held-out test set
# only as an engineering comparison for this prototype.
# For a publication-quality study, model selection should ideally happen
# entirely within cross-validation, leaving the test set untouched until final evaluation.
if tuned_metrics["macro_f1"] >= test_metrics["macro_f1"]:
    FINAL_MODEL = tuned_rf
    FINAL_MODEL_NAME = "Tuned Random Forest"
else:
    FINAL_MODEL = candidate_pipeline
    FINAL_MODEL_NAME = candidate_name

print("Final prototype model:", FINAL_MODEL_NAME)


## 39. Final training strategy

For deployment, the final selected pipeline can be retrained on all available labeled data.

This step is appropriate **after** the model-selection process is finished. It allows the production model to learn from all currently available labeled observations.

The original held-out test metrics should remain recorded as the final evaluation of the selection process.


In [ ]:
# ============================================================
# 39. REFIT FINAL MODEL ON ALL LABELED DATA
# ============================================================

FINAL_MODEL.fit(X, y)

print("Final model refitted on all available labeled observations.")


## 40. Save the model

The complete Scikit-learn pipeline is saved, not only the classifier.

This is important because the model depends on preprocessing steps such as:

- missing-value imputation;
- scaling;
- one-hot encoding.

Saving the complete pipeline prevents training-time preprocessing from being forgotten during inference.


In [ ]:
# ============================================================
# 40. SAVE FINAL MODEL
# ============================================================

MODEL_PATH = MODELS_DIR / "netpredict_ngaoundere_pipeline.joblib"
METRICS_PATH = REPORTS_DIR / "model_metrics.json"
FEATURES_PATH = REPORTS_DIR / "feature_metadata.json"

joblib.dump(FINAL_MODEL, MODEL_PATH)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": FINAL_MODEL_NAME,
            "test_metrics": test_metrics,
            "tuned_random_forest_test_metrics": tuned_metrics,
            "random_state": RANDOM_STATE,
            "test_size": TEST_SIZE,
            "target": TARGET
        },
        f,
        indent=2,
        ensure_ascii=False
    )

with open(FEATURES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "target": TARGET,
            "features": list(X.columns),
            "classes": sorted(y.unique().tolist())
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved model:", MODEL_PATH)
print("Saved metrics:", METRICS_PATH)
print("Saved metadata:", FEATURES_PATH)


# 41. Reusable prediction function

The next function demonstrates how another Python program can use the trained model.

The input must contain the same feature columns used during training. The pipeline handles the preprocessing automatically.


In [ ]:
# ============================================================
# 41. REUSABLE PREDICTION FUNCTION
# ============================================================

def predict_internet_quality(input_data, model=FINAL_MODEL):
    """
    Predict Internet connection quality.

    Parameters
    ----------
    input_data : pandas.DataFrame
        DataFrame containing the same feature columns used during training.

    model : sklearn Pipeline
        Trained preprocessing + model pipeline.

    Returns
    -------
    numpy.ndarray
        Predicted quality labels.
    """
    missing = set(X.columns) - set(input_data.columns)

    if missing:
        raise ValueError(
            f"Missing required features: {sorted(missing)}"
        )

    return model.predict(input_data[X.columns])


# Example using one real test observation:
example = X_test.iloc[[0]].copy()
example_prediction = predict_internet_quality(example)

print("Predicted quality:", example_prediction[0])
print("Actual quality   :", y_test.iloc[0])


## 42. Prediction probabilities

For models that support `predict_proba`, the application can display not only the predicted class but also the model's estimated probability distribution.

These are **model probabilities**, not guaranteed real-world probabilities.


In [ ]:
# ============================================================
# 42. PREDICTION PROBABILITIES
# ============================================================

if hasattr(FINAL_MODEL, "predict_proba"):
    proba = FINAL_MODEL.predict_proba(example)
    classes = FINAL_MODEL.classes_

    probability_df = pd.DataFrame({
        "class": classes,
        "model_probability": proba[0]
    }).sort_values("model_probability", ascending=False)

    display(probability_df)
else:
    print("Final model does not expose predict_proba().")


# 43. Simulated monitoring layer

The original project goal was related to changing connection quality across areas and days.

A future operational system could collect repeated observations and produce a monitoring table like:

- zone;
- timestamp;
- reported quality;
- predicted quality;
- problem frequency;
- number of observations;
- anomaly indicators.

The current survey is not necessarily large enough to justify a real-time monitoring claim, so the following section demonstrates the architecture rather than pretending that the survey itself is a live network feed.


In [ ]:
# ============================================================
# 43. MONITORING-STYLE SUMMARY
# ============================================================

monitoring = df_clean.copy()

if LOCATION_COL is not None:
    monitoring_summary = (
        monitoring.groupby(LOCATION_COL)[TARGET]
        .agg(
            observations="count",
            most_common_quality=lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan
        )
        .reset_index()
        .sort_values("observations", ascending=False)
    )

    display(monitoring_summary)
else:
    print("Monitoring summary requires a detected location field.")


# 44. Future real-world data collection protocol

The current survey can become much more valuable if the next data collection campaign includes objective measurements.

For each observation, record:

| Field | Example |
|---|---|
| timestamp | 2026-09-16 10:30 |
| latitude/longitude | approximate or privacy-safe area |
| zone | Area name |
| provider | Operator |
| technology | 4G / 5G / Wi-Fi |
| download_mbps | 12.4 |
| upload_mbps | 4.8 |
| latency_ms | 52 |
| jitter_ms | 8 |
| packet_loss_pct | 1.2 |
| signal_strength_dbm | -83 |
| connection_quality | derived target |

This would allow a future version to move from **survey-based quality classification** toward genuine **network-performance prediction**.

### Recommended future targets

1. `download_mbps` — regression
2. `latency_ms` — regression
3. `connection_quality` — classification
4. `network_problem` — classification
5. `anomaly` — anomaly detection

With repeated measurements over weeks or months, a time-series forecasting extension could also be studied.


# 45. Possible V2 architecture

```text
                 ┌─────────────────────┐
                 │ Field Data Collection│
                 │ Survey + Speed Tests │
                 └──────────┬──────────┘
                            │
                            ▼
                 ┌─────────────────────┐
                 │ Data Cleaning        │
                 │ Validation           │
                 └──────────┬──────────┘
                            │
                            ▼
                 ┌─────────────────────┐
                 │ Feature Engineering  │
                 └──────────┬──────────┘
                            │
             ┌──────────────┼──────────────┐
             ▼              ▼              ▼
        Classification   Regression    Anomaly Detection
             │              │              │
             └──────────────┼──────────────┘
                            ▼
                 ┌─────────────────────┐
                 │ Prediction API       │
                 └──────────┬──────────┘
                            ▼
                 ┌─────────────────────┐
                 │ Streamlit Dashboard  │
                 │ Maps / Trends / ML   │
                 └─────────────────────┘
```


# 46. Limitations

This project has several important limitations.

### 1. Small sample size

The provided dataset contains approximately 100 observations. This is useful for a student prototype and exploratory study, but it is small for a robust predictive system.

### 2. Survey bias

Responses represent reported user experiences. They are not equivalent to continuous technical network measurements.

### 3. Geographic coverage

The sampled locations may not represent every neighborhood or every Internet user in Ngaoundéré.

### 4. Temporal coverage

A limited collection period cannot establish long-term seasonal or yearly network behavior.

### 5. Causality

Feature importance and statistical associations do not prove that a variable causes poor Internet quality.

### 6. External validation

The model should be evaluated on a completely new dataset collected later and, ideally, under a predefined measurement protocol.


# 47. Ethical and privacy considerations

If the original survey contains personal information, such as names, telephone numbers, email addresses or precise addresses, those fields **must not be uploaded to a public GitHub repository**.

Before publication:

- anonymize or remove personal identifiers;
- avoid exposing precise residential locations;
- review metadata and exported notebooks;
- do not publish raw responses containing sensitive information;
- keep only the variables necessary for the research objective.

The public repository should preferably contain a sanitized dataset or a documented sample rather than identifiable raw survey responses.


# 48. Reproducibility checklist

Before publishing this project on GitHub:

- [ ] Raw data is sanitized.
- [ ] No personal identifiers are public.
- [ ] `requirements.txt` is included.
- [ ] Random seeds are fixed.
- [ ] Data preprocessing is reproducible.
- [ ] Target definition is documented.
- [ ] Train/test split is documented.
- [ ] Cross-validation strategy is documented.
- [ ] Metrics are reported honestly.
- [ ] Limitations are stated.
- [ ] Saved model is versioned.
- [ ] README explains how to run the notebook.
- [ ] Streamlit application is tested separately.


# 49. Suggested `requirements.txt`

The GitHub repository can use a file containing:

```text
pandas
numpy
matplotlib
seaborn
scikit-learn
joblib
jupyter
streamlit
```

Pin exact versions later after testing the final environment.


# 50. Suggested GitHub README summary

## NetPredict-Ngaoundéré

**NetPredict-Ngaoundéré** is a machine learning project focused on predicting reported Internet connection quality using a locally collected survey dataset from Ngaoundéré, Cameroon.

The project follows an end-to-end supervised learning workflow, including data cleaning, exploratory analysis, feature engineering, preprocessing, model comparison, cross-validation, evaluation, error analysis and model persistence.

A future application layer can use the trained pipeline to provide interactive Internet-quality predictions and network-quality analytics.

### Key technologies

- Python
- Pandas
- NumPy
- Scikit-learn
- Matplotlib
- Seaborn
- Joblib
- Jupyter Notebook
- Streamlit

### Important limitation

The current dataset primarily contains survey responses. Therefore, the project predicts **reported Internet connection quality**, not a directly measured download speed in Mbps. Future versions should combine survey responses with objective speed-test and network-performance measurements.


# 51. Final project conclusion

This notebook demonstrates how a locally collected real-world dataset can be transformed into a complete supervised machine learning workflow.

The project starts from a practical problem observed in Ngaoundéré: Internet quality is not constant and may vary across context and location. Instead of relying only on anecdotal observations, the project structures collected data and uses machine learning to identify patterns and generate predictions.

The current version should be considered a **proof of concept**.

Its strongest future direction is the creation of a larger, longitudinal dataset combining:

- user-reported quality;
- download/upload speed;
- latency;
- packet loss;
- signal strength;
- provider;
- technology;
- location;
- timestamp.

Such a dataset would allow NetPredict-Ngaoundéré to evolve from a survey-based classification project into a broader **Internet Network Quality Analytics and Prediction Platform**.

---

**Author:** Nasser Aminou  
**Project:** NetPredict-Ngaoundéré  
**Location:** Ngaoundéré, Cameroon
